In [2]:
import sys
from pathlib import Path
import pandas as pd

src_dir = Path.cwd().parent

# sys.path strictly for importing modules
sys.path.append(str(src_dir))
from utils.data_utils import *

HOSP_DIR = src_dir / "data" / "mimic-iv" / "hosp"

In [3]:
diabetic_patients = load_data(src_dir / "data" / "processed" / "diabetic_patients.csv.gz")
diabetic_patients.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000635,20642640,7,E119,10
1,10000980,20897796,5,E1122,10
2,10001176,23334588,2,25000,9
3,10001843,21728396,9,E119,10
4,10001877,21320596,6,25000,9


In [4]:
diabetic_ids = diabetic_patients["subject_id"].unique()
diabetic_ids

array([10000635, 10000980, 10001176, ..., 19999287, 19999379, 19999828])

In [5]:
# Define all file paths for cohort extraction
patients_path = HOSP_DIR / "patients.csv.gz"
admissions_path = HOSP_DIR / "admissions.csv.gz"
labevents_path = HOSP_DIR / "labevents.csv.gz"
prescriptions_path = HOSP_DIR / "prescriptions.csv.gz"
diagnoses_icd_path = HOSP_DIR / "diagnoses_icd.csv.gz"
procedures_icd_path = HOSP_DIR / "procedures_icd.csv.gz"

In [6]:
# Load data
patients = load_data(patients_path)
admissions = load_data(admissions_path)
# labevents = load_data(labevents_path)
# prescriptions = load_data(prescriptions_path)
diagnoses_icd = load_data(diagnoses_icd_path)
# procedures_icd = load_data(procedures_icd_path)

# Filter data for diabetic patients
patients = filter_to_cohort(patients, diabetic_ids)
admissions = filter_to_cohort(admissions, diabetic_ids)
# labevents = filter_to_cohort(labevents, diabetic_ids)
# prescriptions = filter_to_cohort(prescriptions, diabetic_ids)
diagnoses_icd = filter_to_cohort(diagnoses_icd, diabetic_ids)
# procedures_icd = filter_to_cohort(procedures_icd, diabetic_ids)

In [8]:
diagnoses_icd.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
88,10000635,20642640,1,R0789,10
89,10000635,20642640,2,R29810,10
90,10000635,20642640,3,R200,10
91,10000635,20642640,4,I10,10
92,10000635,20642640,5,E7800,10


In [ ]:
patients_summary = patients[["subject_id", "gender", "anchor_age"]].rename(columns={"anchor_age": "age"})
admissions_summary = (
    admissions.groupby("subject_id")
    .agg(
        n_admissions=("hadm_id", "nunique"),
        first_admission_date=("admittime", "min"),
        last_admission_date=("admittime", "max"),
    )
    .reset_index()
)
HTN_CODES = ["I10", "401"]

admissions_summary.head()

,subject_id,n_admissions,first_admission_date,last_admission_date
0,10000635,2,2136-06-19 14:24:00,2143-12-23 14:55:00
1,10000980,7,2188-01-03 17:41:00,2193-08-15 01:01:00
2,10001176,1,2186-11-29 03:56:00,2186-11-29 03:56:00
3,10001843,2,2131-11-09 16:05:00,2134-12-05 00:10:00
4,10001877,2,2149-05-21 15:53:00,2150-11-21 23:02:00
